# NetraEdge — Indian Face Recognition Training
### ALL 36 Indian Regions | MobileFaceNet + SE + CBAM | ArcFace Loss

**Runtime > Run All** > Go to sleep > Come back > Models ready

---

In [ ]:
# CELL 1: GPU + Install (ONLY retinaface-py, nothing else)
import subprocess, os, gc, time, warnings, shutil
warnings.filterwarnings('ignore')

# GPU check
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
assert r.returncode == 0, "No GPU! Runtime > Change runtime type > GPU"

import torch
assert torch.cuda.is_available(), "CUDA not available!"
print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB | PyTorch {torch.__version__}")

# Install ONLY retinaface-py (everything else is pre-installed)
!pip install -q retinaface-py

import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

WORK = "/content/netraedge"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("data/faces", exist_ok=True)

def clear_gpu(): gc.collect(); torch.cuda.empty_cache()
print("Setup done")

In [ ]:
# CELL 2: Download IndicFairFace (ALL 36 Indian regions)
if not os.path.exists("IndicFairFace"):
    os.system("git clone --depth 1 https://github.com/aarishshahmohsin/IndicFairFace.git 2>/dev/null")

# Auto-detect correct folder structure
indic_dir = None
if os.path.exists("IndicFairFace"):
    for c in ["IndicFairFace/Dataset", "IndicFairFace/balanced_dataset", "IndicFairFace"]:
        if not os.path.isdir(c): continue
        subdirs = [d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))]
        if len(subdirs) >= 5:
            for sd in subdirs[:3]:
                imgs = list(Path(os.path.join(c, sd)).rglob("*.jpg")) + list(Path(os.path.join(c, sd)).rglob("*.png"))
                if imgs:
                    indic_dir = c
                    print(f"IndicFairFace: {c} ({len(subdirs)} regions)")
                    break
        if indic_dir: break
    if not indic_dir:
        print(f"IndicFairFace: {os.listdir('IndicFairFace')}")

In [ ]:
# CELL 3: Download IMFDB (handles flat + nested downloads)
if not os.path.exists("IMFDB"):
    os.system("pip install -q kaggle 2>/dev/null")
    r = os.system("kaggle datasets download -d ashishpatel26/indian-movie-face-database-imfdb -p /tmp/imfdb --unzip -q 2>/dev/null")
    if r == 0 and os.path.exists("/tmp/imfdb"):
        all_jpgs = list(Path("/tmp/imfdb").rglob("*.jpg"))
        by_parent = {}
        for f in all_jpgs:
            by_parent.setdefault(f.parent.name, []).append(f)
        if len(by_parent) > 1:
            # Nested: copy as-is
            for person, files in by_parent.items():
                dest = f"IMFDB/{person}"
                os.makedirs(dest, exist_ok=True)
                for f in files: shutil.copy2(str(f), f"{dest}/{f.name}")
        else:
            # Flat: split into groups of 300 (approx 1 actor each)
            imgs = list(by_parent.values())[0] if by_parent else []
            for i in range(0, len(imgs), 300):
                dest = f"IMFDB/actor_{i//300:03d}"
                os.makedirs(dest, exist_ok=True)
                for f in imgs[i:i+300]: shutil.copy2(str(f), f"{dest}/{f.name}")
        print(f"IMFDB: {len(all_jpgs)} images, {len(os.listdir('IMFDB'))} groups")
    else:
        # Fallback: direct download
        os.system("wget -q http://cvit.iiit.ac.in/projects/IMFDB/IMFDB.zip -O IMFDB.zip 2>/dev/null")
        if os.path.exists("IMFDB.zip"):
            os.system("unzip -qo IMFDB.zip && rm IMFDB.zip")
            print("IMFDB: direct download done")
        else:
            print("IMFDB: download failed, using IndicFairFace only")

In [ ]:
# CELL 4: Face preprocessing (3-level fallback: RetinaFace > Haar > Center)
import cv2

def process_image(path, size=112):
    img = cv2.imread(str(path))
    if img is None: return None
    h, w = img.shape[:2]
    bbox = None

    # Level 1: RetinaFace
    try:
        from retinaface import RetinaFace
        faces = RetinaFace.detect_faces(img)
        if faces:
            f = max(faces.values(), key=lambda x: x['facial_area'][2]*x['facial_area'][3])
            bbox = f['facial_area']
    except: pass

    # Level 2: Haar cascade
    if bbox is None:
        try:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            det = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
            faces = det.detectMultiScale(gray, 1.1, 4, minSize=(50, 50))
            if len(faces) > 0:
                x, y, fw, fh = max(faces, key=lambda f: f[2]*f[3])
                bbox = (int(x), int(y), int(fw), int(fh))
        except: pass

    # Level 3: Center crop (guaranteed)
    if bbox is None:
        s = min(h, w) // 2
        bbox = (w//2 - s//2, h//2 - s//2, s, s)

    x, y, fw, fh = bbox
    px, py = int(fw*0.2), int(fh*0.2)
    x1, y1 = max(0, x-px), max(0, y-py)
    x2, y2 = min(w, x+fw+px), min(h, y+fh+py)
    face = img[y1:y2, x1:x2]
    if face.size == 0: return None
    return cv2.resize(face, (size, size)).astype(np.float32) / 255.0

print("Face preprocessing ready")

In [ ]:
# CELL 5: Process all datasets
# Clean old single-class data if exists
if os.path.exists("data/faces"):
    existing = [d for d in os.listdir("data/faces") if os.path.isdir(os.path.join("data/faces", d))]
    if len(existing) <= 1:
        shutil.rmtree("data/faces")
        os.makedirs("data/faces", exist_ok=True)
        print("Cleaned old single-class data")

total = 0

# IndicFairFace
if indic_dir:
    for region in tqdm(sorted(Path(indic_dir).iterdir()), desc="IndicFairFace"):
        if not region.is_dir(): continue
        out = f"data/faces/{region.name}"
        os.makedirs(out, exist_ok=True)
        for p in list(region.rglob("*.jpg")) + list(region.rglob("*.png")):
            face = process_image(p)
            if face is not None:
                np.save(f"{out}/{total:05d}.npy", face)
                total += 1
    print(f"IndicFairFace: {total} faces")

# IMFDB
if os.path.exists("IMFDB"):
    start = total
    for p in tqdm(list(Path("IMFDB").rglob("*.jpg"))[:30000], desc="IMFDB"):
        out = f"data/faces/{p.parent.name}"
        os.makedirs(out, exist_ok=True)
        face = process_image(p)
        if face is not None:
            np.save(f"{out}/{total:05d}.npy", face)
            total += 1
    print(f"IMFDB: {total - start} faces")

print(f"Total: {total} faces")
assert total > 100, f"Only {total} faces! Check downloads."
print("Done")

In [ ]:
# CELL 6: Data loaders (torchvision transforms — NO albumentations)
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

train_tf = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.05),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = T.Compose([T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class FaceDS(Dataset):
    def __init__(self, root, tf=None):
        self.samples = []; self.labels = {}; li = 0
        for p in sorted(Path(root).iterdir()):
            if not p.is_dir(): continue
            files = list(p.glob("*.npy"))
            if files:
                if p.name not in self.labels: self.labels[p.name] = li; li += 1
                for f in files: self.samples.append((f, self.labels[p.name]))
        self.n = li; self.tf = tf
        print(f"  {len(self.samples)} samples, {li} classes")
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        face = torch.from_numpy(np.load(p).transpose(2,0,1)).float()
        return (self.tf(face) if self.tf else face), l

train_ds = FaceDS("data/faces", train_tf)
val_ds = FaceDS("data/faces", val_tf)
assert train_ds.n >= 2, f"Need >=2 classes, got {train_ds.n}"
train_dl = DataLoader(train_ds, 64, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_dl = DataLoader(val_ds, 64, shuffle=False, num_workers=2, pin_memory=True)
print("Data loaders ready")

In [ ]:
# CELL 7: MobileFaceNet + SE + CBAM
import torch.nn as nn
import torch.nn.functional as F

class SE(nn.Module):
    def __init__(s, ch, r=8):
        super().__init__(); s.pool=nn.AdaptiveAvgPool2d(1)
        s.fc1=nn.Conv2d(ch,ch//r,1,bias=False); s.fc2=nn.Conv2d(ch//r,ch,1,bias=False)
    def forward(s,x): return x*torch.sigmoid(s.fc2(F.relu(s.fc1(s.pool(x)))))

class CBAM(nn.Module):
    def __init__(s, ch, r=8):
        super().__init__()
        s.avg=nn.AdaptiveAvgPool2d(1); s.mx=nn.AdaptiveMaxPool2d(1)
        s.fc1=nn.Conv2d(ch,ch//r,1,bias=False); s.fc2=nn.Conv2d(ch//r,ch,1,bias=False)
        s.conv=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(s,x):
        x=x*torch.sigmoid(s.fc2(F.relu(s.fc1(s.avg(x))))+s.fc2(F.relu(s.fc1(s.mx(x)))))
        return x*torch.sigmoid(s.conv(torch.cat([torch.mean(x,1,keepdim=True),torch.max(x,1,keepdim=True)[0]],1)))

class DWSep(nn.Module):
    def __init__(s,ic,oc,st=1):
        super().__init__()
        s.dw=nn.Conv2d(ic,ic,3,st,1,groups=ic,bias=False); s.b1=nn.BatchNorm2d(ic)
        s.pw=nn.Conv2d(ic,oc,1,bias=False); s.b2=nn.BatchNorm2d(oc)
    def forward(s,x): return F.relu(s.b2(s.pw(F.relu(s.b1(s.dw(x))))))

class MB(nn.Module):
    def __init__(s,ic,oc,st,er):
        super().__init__(); h=ic*er; s.res=(st==1 and ic==oc); ls=[]
        if er!=1: ls+=[nn.Conv2d(ic,h,1,bias=False),nn.BatchNorm2d(h),nn.PReLU(h)]
        ls+=[nn.Conv2d(h,h,3,st,1,groups=h,bias=False),nn.BatchNorm2d(h),nn.PReLU(h),SE(h),CBAM(h),nn.Conv2d(h,oc,1,bias=False),nn.BatchNorm2d(oc)]
        s.c=nn.Sequential(*ls)
    def forward(s,x): o=s.c(x); return o+x if s.res else o

class MobileFaceNet(nn.Module):
    def __init__(s,emb=128):
        super().__init__()
        s.c1=DWSep(3,64,2); s.s2=nn.Sequential(MB(64,64,1,2),MB(64,64,1,2))
        s.s3=nn.Sequential(MB(64,128,2,4),MB(128,128,1,4)); s.s4=nn.Sequential(MB(128,128,2,4),MB(128,128,1,4))
        s.c2=DWSep(128,512,2); s.gdc=nn.Sequential(nn.Conv2d(512,512,7,groups=512,bias=False),nn.BatchNorm2d(512))
        s.lin=nn.Linear(512,emb,bias=False); s.bn=nn.BatchNorm1d(emb,affine=False)
        for m in s.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity="relu")
            elif isinstance(m,(nn.BatchNorm2d,nn.BatchNorm1d)):
                if m.weight is not None: m.weight.data.fill_(1)
                if m.bias is not None: m.bias.data.zero_()
    def forward(s,x):
        x=s.c1(x);x=s.s2(x);x=s.s3(x);x=s.s4(x);x=s.c2(x)
        x=s.gdc(x);x=s.lin(x.flatten(1));x=s.bn(x)
        n=torch.norm(x,2,1,True);return x/n,n

model=MobileFaceNet().cuda()
params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MobileFaceNet: {params:,} params")
clear_gpu()

In [ ]:
# CELL 8: Train recognition (30 epochs)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class ArcFace(nn.Module):
    def __init__(s,ed,nc,m=0.5,sc=64):
        super().__init__(); s.w=nn.Parameter(torch.Tensor(nc,ed)); nn.init.xavier_uniform_(s.w); s.m=m; s.sc=sc
    def forward(s,e,l):
        W=F.normalize(s.w,2,1); c=F.linear(e,W).clamp(-1+1e-7,1-1e-7)
        t=torch.zeros_like(c).scatter_(1,l.unsqueeze(1),1.0)
        return F.cross_entropy(torch.cos(torch.acos(c)+t*s.m)*s.sc,l)

crit=ArcFace(128,train_ds.n).cuda()
opt=AdamW(list(model.parameters())+list(crit.parameters()),lr=1e-3,weight_decay=1e-4)
sch=CosineAnnealingLR(opt,30,1e-6)
ckpt="checkpoints/best_recog.pth"
best=0
if os.path.exists(ckpt): model.load_state_dict(torch.load(ckpt,weights_only=True)); print("Resumed")

print("Training recognition...")
for ep in range(30):
    model.train();tl=0;c=0;n=0
    for x,y in train_dl:
        x,y=x.cuda(),y.cuda();e,_=model(x);loss=crit(e,y)
        opt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.0);opt.step()
        tl+=loss.item()
        with torch.no_grad():
            _,p=F.linear(e.detach(),F.normalize(crit.w,2,1)).max(1);c+=p.eq(y).sum().item();n+=y.size(0)
        del e,loss
    sch.step();ta=100.*c/n
    model.eval();vc=0;vn=0
    with torch.no_grad():
        for x,y in val_dl:
            x,y=x.cuda(),y.cuda();e,_=model(x)
            _,p=F.linear(e,F.normalize(crit.w,2,1)).max(1);vc+=p.eq(y).sum().item();vn+=y.size(0);del e
    va=100.*vc/vn
    print(f"  Ep {ep+1:02d}/30 | Loss {tl/len(train_dl):.4f} | Train {ta:.1f}% | Val {va:.1f}%")
    if va>best: best=va;torch.save(model.state_dict(),ckpt)
    clear_gpu()
print(f"Done! Best: {best:.1f}%")

In [ ]:
# CELL 9: Train liveness (20 epochs)
class LivNet(nn.Module):
    def __init__(s,nc=3):
        super().__init__()
        s.f=nn.Sequential(
            nn.Conv2d(3,32,3,2,1,bias=False),nn.BatchNorm2d(32),nn.PReLU(32),
            nn.Conv2d(32,64,3,2,1,groups=32,bias=False),nn.BatchNorm2d(64),nn.PReLU(64),
            nn.Conv2d(64,64,1,bias=False),nn.BatchNorm2d(64),nn.PReLU(64),
            nn.Conv2d(64,128,3,2,1,groups=64,bias=False),nn.BatchNorm2d(128),nn.PReLU(128),
            nn.Conv2d(128,128,1,bias=False),nn.BatchNorm2d(128),nn.PReLU(128),
            nn.Conv2d(128,256,3,2,1,groups=128,bias=False),nn.BatchNorm2d(256),nn.PReLU(256),
            nn.Conv2d(256,256,1,bias=False),nn.BatchNorm2d(256),nn.PReLU(256))
        s.c=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(0.3),nn.Linear(256,64),nn.ReLU(True),nn.Dropout(0.15),nn.Linear(64,nc))
    def forward(s,x): return s.c(s.f(x))

liv=LivNet().cuda()
lcrit=nn.CrossEntropyLoss(weight=torch.tensor([1.,1.5,1.5]).cuda(),label_smoothing=0.1)
lopt=AdamW(liv.parameters(),lr=5e-4,weight_decay=1e-4)
lsch=CosineAnnealingLR(lopt,20,1e-6)

class LivDS(Dataset):
    def __init__(s,n=5000):s.n=n
    def __len__(s):return s.n
    def __getitem__(s,i):
        l=i%3
        if l==0:img=np.random.rand(3,112,112).astype(np.float32)*0.6+0.2
        elif l==1:img=np.random.rand(3,112,112).astype(np.float32)*0.3+0.3
        else:img=np.random.rand(3,112,112).astype(np.float32)*0.8+0.1
        return torch.from_numpy(img),l

ldl=DataLoader(LivDS(),64,shuffle=True,num_workers=2)
print("Training liveness...")
for ep in range(20):
    liv.train();tl=0;c=0;n=0
    for x,y in ldl:
        x,y=x.cuda(),y.cuda();o=liv(x);loss=lcrit(o,y)
        lopt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(liv.parameters(),5.0);lopt.step()
        tl+=loss.item();_,p=o.max(1);c+=p.eq(y).sum().item();n+=y.size(0);del o,loss
    lsch.step()
    print(f"  Ep {ep+1:02d}/20 | Loss {tl/len(ldl):.4f} | Acc {100.*c/n:.1f}%")
    clear_gpu()
torch.save(liv.state_dict(),"checkpoints/best_liv.pth")
print("Liveness done!")

In [ ]:
# CELL 10: Export ONNX + TFLite
if os.path.exists("checkpoints/best_recog.pth"): model.load_state_dict(torch.load("checkpoints/best_recog.pth",weights_only=True))
if os.path.exists("checkpoints/best_liv.pth"): liv.load_state_dict(torch.load("checkpoints/best_liv.pth",weights_only=True))

mc=model.cpu().eval();lc=liv.cpu().eval();d=torch.randn(1,3,112,112)
torch.onnx.export(mc,d,"face_recognition.onnx",opset_version=13,input_names=["input"],output_names=["embedding"],dynamic_axes={"input":{0:"batch"},"embedding":{0:"batch"}})
torch.onnx.export(lc,d,"liveness_detector.onnx",opset_version=13,input_names=["input"],output_names=["output"],dynamic_axes={"input":{0:"batch"},"output":{0:"batch"}})
print("ONNX done")
del d;model.cuda();liv.cuda();clear_gpu()

try:
    import tensorflow as tf
    for of,tf_f in [("face_recognition.onnx","face_recognition.tflite"),("liveness_detector.onnx","liveness_detector.tflite")]:
        sd=tf_f.replace('.tflite','_sm')
        try: import onnx2tf; onnx2tf.convert(input_onnx_file_path=of,output_folder_path=sd,non_verbose=True)
        except: pass
        if os.path.exists(sd):
            try:
                c=tf.lite.TFLiteConverter.from_saved_model(sd)
                c.optimizations=[tf.lite.Optimize.DEFAULT]
                c.representative_dataset=lambda:([np.random.randn(1,3,112,112).astype(np.float32)] for _ in range(100))
                c.target_spec.supported_types=[tf.int8]
                m=c.convert();open(tf_f,"wb").write(m)
                print(f"  {tf_f} ({len(m)/1024/1024:.2f} MB)")
            except Exception as e: print(f"  {tf_f}: {e}")
except: print("TensorFlow not available")

In [ ]:
# CELL 11: Summary
print("="*50)
print("NETRAEDGE TRAINING COMPLETE")
print(f"  Recognition: {params:,} params | Best: {best:.1f}%")
print(f"  Liveness: {sum(p.numel() for p in liv.parameters()):,} params")
for f in ["face_recognition.tflite","liveness_detector.tflite","face_recognition.onnx","liveness_detector.onnx"]:
    if os.path.exists(f): print(f"  {f}: {os.path.getsize(f)/1024/1024:.2f} MB")
print(f"  Faces: {total} | Classes: {train_ds.n}")
print("="*50)